In [2]:
import pandas as pd
import numpy as np

In [6]:
# load transaction data
df_transaction = pd.read_csv('/Users/mariangarcia/Downloads/archive (2)/transaction_data.csv')
# check number of rows in the transaction data
rows_transaction = len(df_transaction)
print(f'The number of rows in the transaction data is: {rows_transaction}')

# load product data
df_product = pd.read_csv('/Users/mariangarcia/Downloads/archive (2)/product.csv')
rows_product = len(df_product)
print(f'The number of rows in the product data is: {rows_product}')

The number of rows in the transaction data is: 2595732
The number of rows in the product data is: 92353


In [ ]:
# check for missing values in the transaction data
df_transaction.isna().sum()

# you're gonna get the number of missing values in each column of the transaction data and print it out. You can use the isna() method to check for missing values and the sum() method to count them.

##

household_key        0
BASKET_ID            0
DAY                  0
PRODUCT_ID           0
QUANTITY             0
SALES_VALUE          0
STORE_ID             0
RETAIL_DISC          0
TRANS_TIME           0
WEEK_NO              0
COUPON_DISC          0
COUPON_MATCH_DISC    0
dtype: int64

In [ ]:
# check for missing values in the product data
df_product.isna().sum()


PRODUCT_ID              0
MANUFACTURER            0
DEPARTMENT              0
BRAND                   0
COMMODITY_DESC          0
SUB_COMMODITY_DESC      0
CURR_SIZE_OF_PRODUCT    0
dtype: int64

In [10]:
df_product["DEPARTMENT"].value_counts()

DEPARTMENT
GROCERY            39021
DRUG GM            31529
PRODUCE             3118
COSMETICS           3011
NUTRITION           2914
MEAT                2544
MEAT-PCKGD          2427
DELI                2354
PASTRY              2149
FLORAL               938
SEAFOOD-PCKGD        563
MISC. TRANS.         490
SPIRITS              377
SEAFOOD              369
GARDEN CENTER        128
RESTAURANT           102
MISC SALES TRAN       88
SALAD BAR             48
COUP/STR & MFG        39
TRAVEL & LEISUR       28
FROZEN GROCERY        23
KIOSK-GAS             16
                      15
CHEF SHOPPE           14
RX                     9
CNTRL/STORE SUP        4
DAIRY DELI             3
TOYS                   3
POSTAL CENTER          3
GM MERCH EXP           3
VIDEO RENTAL           3
AUTOMOTIVE             2
PHOTO                  2
VIDEO                  2
PROD-WHS SALES         2
DELI/SNACK BAR         2
CHARITABLE CONT        2
GRO BAKERY             2
PORK                   1
HBC           

In [ ]:
# merge the two dataframes on the PRODUCT_ID column
df = df_transaction.merge(df_product, on="PRODUCT_ID", how="left")
print(len(df_transaction), len(df))   # these should match, safety check making sure we didn't lose any rows in the merge
df.groupby("DEPARTMENT")["SALES_VALUE"].sum().sort_values(ascending=False).head(15)

2595732 2595732


DEPARTMENT
GROCERY            4093814.14
DRUG GM            1055358.03
PRODUCE             557452.11
MEAT                548786.81
KIOSK-GAS           544222.28
MEAT-PCKGD          412436.77
DELI                260866.51
PASTRY              121739.86
MISC SALES TRAN     119960.04
NUTRITION            97669.04
SEAFOOD-PCKGD        63093.62
FLORAL               39653.34
COSMETICS            32360.37
SALAD BAR            30953.30
SEAFOOD              29496.92
Name: SALES_VALUE, dtype: float64

In [12]:
# checking the top 10 commodities in the MISC SALES TRAN department
df[df["DEPARTMENT"] == "MISC SALES TRAN"]["COMMODITY_DESC"].value_counts().head(10)

COMMODITY_DESC
COUPON/MISC ITEMS    6050
Name: count, dtype: int64

In [13]:
# exclude the KIOSK-GAS and MISC SALES TRAN departments from the analysis
before = len(df)
df_clean = df[~df["DEPARTMENT"].isin(["KIOSK-GAS", "MISC SALES TRAN"])]
print(before, len(df_clean), before - len(df_clean))

2595732 2567623 28109


## Cleaning Log

### 1. Excluded KIOSK-GAS department
- Rows removed: 22,059 transaction rows (~0.8% of rows)
- Sales removed: $544,222 (~6.8% of total sales)
- Reason: Gas is not a grocery product and doesn't respond to
  promotions the way groceries do. Including it would blur the
  discount and promotion results.
- Note: It's only 16 products, but it's a large share of sales because
  each gas purchase is a big dollar amount. That's why it was excluded
  despite the small row count.

### 2. Excluded MISC SALES TRAN department
- Rows removed: 6,050 transaction rows
- Content: all rows have COMMODITY_DESC = "COUPON/MISC ITEMS"
- Sales removed: $119,960 (~1.5% of total sales)
- Reason: These are coupon and miscellaneous bookkeeping entries, not
  products customers chose to buy, so they don't belong in a product
  or discount analysis.
- Caveat: Coupons are part of this project. They'll be analyzed
  separately using coupon_redempt and the COUPON_DISC columns.

### Summary after steps 1-2
- Rows before: 2,595,732 | Rows after: 2,567,623
- Total removed: 28,109 rows (~1.1% of the data)
- Merge check: transaction and merged row counts matched (no duplicates)

In [14]:
# check how many rows were excluded from the KIOSK-GAS department
(df["DEPARTMENT"] == "KIOSK-GAS").sum()

22059